# RetrainingFromCheckpoints

## Importing required libraries

In [1]:
import pandas as pd

## Loading the Data

In [2]:
df = pd.read_csv('Datasets/CarPrice_Assignment.csv')

## Data Cleaning and Preparation

### Check the dimensions 

In [3]:
df.shape

(205, 26)

### Check columns names

In [4]:
df.columns

Index(['car_ID', 'symboling', 'CarName', 'fueltype', 'aspiration',
       'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase',
       'carlength', 'carwidth', 'carheight', 'curbweight', 'enginetype',
       'cylindernumber', 'enginesize', 'fuelsystem', 'boreratio', 'stroke',
       'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg',
       'price'],
      dtype='object')

In [5]:
df.head(5)

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


### Check for Nulls

In [6]:
df.isnull().values.any()

False

### Drop some columns

In [7]:
df.drop(['car_ID', 'symboling', 'CarName'], axis=1, inplace=True)

In [8]:
df.shape

(205, 23)

### Get Categoricals

In [9]:
df = pd.get_dummies(df)

In [10]:
df.head()

,wheelbase,carlength,carwidth,carheight,curbweight,enginesize,boreratio,stroke,compressionratio,horsepower,...,cylindernumber_twelve,cylindernumber_two,fuelsystem_1bbl,fuelsystem_2bbl,fuelsystem_4bbl,fuelsystem_idi,fuelsystem_mfi,fuelsystem_mpfi,fuelsystem_spdi,fuelsystem_spfi
0,88.6,168.8,64.1,48.8,2548,130,3.47,2.68,9.0,111,...,0,0,0,0,0,0,0,1,0,0
1,88.6,168.8,64.1,48.8,2548,130,3.47,2.68,9.0,111,...,0,0,0,0,0,0,0,1,0,0
2,94.5,171.2,65.5,52.4,2823,152,2.68,3.47,9.0,154,...,0,0,0,0,0,0,0,1,0,0
3,99.8,176.6,66.2,54.3,2337,109,3.19,3.40,10.0,102,...,0,0,0,0,0,0,0,1,0,0
4,99.4,176.6,66.4,54.3,2824,136,3.19,3.40,8.0,115,...,0,0,0,0,0,0,0,1,0,0


## Split data for simulate retrain from checkpoint

In [11]:
automobile_train = df.head(82)
automobile_test = df.tail(41)

In [12]:
automobile_train.shape, automobile_test.shape

((82, 52), (41, 52))

## Machine Learning: LinearRegression

In [13]:
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

### Define Output and Inputs

In [14]:
X_train = automobile_train.drop(['price'], axis = 1)
y_train = automobile_train['price']

X_test = automobile_test.drop(['price'], axis = 1)
y_test = automobile_test['price']

In [15]:
X_train.shape, X_test.shape

((82, 51), (41, 51))

In [16]:
y_train.shape, y_test.shape

((82,), (41,))

### Instantiate Ensemble Regressor Model

In [17]:
regressor_model = RandomForestRegressor(n_estimators=5, warm_start=True)

### Fit Model

In [18]:
rfr_model = regressor_model.fit(X_train, y_train)

### Score

In [19]:
rfr_model.score(X_train, y_train)

0.9751558746512793

In [20]:
y_pred = rfr_model.predict(X_test)

In [21]:
test_score = r2_score(y_test, y_pred)
test_score

-0.5079911345851809

### Write a checkpoint

In [22]:
rfr_model_param = {}

rfr_model_param['model'] = rfr_model
rfr_model_param['sklearn_version'] = sklearn.__version__
rfr_model_param['r2score'] = test_score

rfr_model_param

{'model': RandomForestRegressor(n_estimators=5, warm_start=True),
 'sklearn_version': '0.24.2',
 'r2score': -0.5079911345851809}

#### Serializing using Joblib

In [23]:
import joblib

In [24]:
filename = 'Models/rfr_model_checkpoint.joblib'

#### Save

In [25]:
joblib.dump(rfr_model_param, filename)

['Models/rfr_model_checkpoint.joblib']

#### Load

In [26]:
joblib_model = joblib.load(filename)

In [27]:
joblib_model

{'model': RandomForestRegressor(n_estimators=5, warm_start=True),
 'sklearn_version': '0.24.2',
 'r2score': -0.5079911345851809}

## Retraining

In [28]:
joblib_model['model'].n_estimators= 15

### Load new data

In [29]:
retrain_data = df

In [30]:
X_train = retrain_data.drop(['price'], axis = 1)
y_train = retrain_data['price']

In [31]:
rfr_retrained_model = joblib_model['model'].fit(X_train, y_train)

In [32]:
rfr_retrained_model.score(X_train, y_train)

0.9362110099846244

In [33]:
y_pred = rfr_retrained_model.predict(X_test)

test_score = r2_score(y_test, y_pred)
test_score

0.8030042473634857

## Save new checkpoint 

In [34]:
rfr_retrained_model_param = {}

rfr_retrained_model_param['model'] = rfr_retrained_model
rfr_retrained_model_param['sklearn_version'] = sklearn.__version__
rfr_retrained_model_param['r2score'] = test_score

rfr_retrained_model_param

{'model': RandomForestRegressor(n_estimators=15, warm_start=True),
 'sklearn_version': '0.24.2',
 'r2score': 0.8030042473634857}

In [35]:
filename = 'Models/rfr_retrained_model_checkpoint.joblib'

In [36]:
joblib.dump(rfr_retrained_model_param, filename)

['Models/rfr_retrained_model_checkpoint.joblib']